<a href="https://colab.research.google.com/github/farukkilinc1903/group6-innovation-campus-samsung/blob/main/Assignment_3_PartA_B_C_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assignment 3**

# **PART A: Model Refinement**

## **1. Overview**

This section focuses on the model refinement phase of the project. In the previous stage, different regression models were compared to predict school electricity access using country-level socio-economic, infrastructure, and satellite-based indicators. The target variable is Electricity_Access_Percent.

After comparing Random Forest, XGBoost, and Gradient Boosting, XGBoost achieved the best overall performance and was selected for refinement. In this phase, GroupShuffleSplit is used for train-test splitting and GroupKFold is used for cross-validation to prevent data leakage across country-year observations.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    GroupShuffleSplit, GroupKFold,
    cross_val_score, GridSearchCV
)
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

# Load datasets
school_df = pd.read_csv("d0.school_electricity_access.csv")
national_electricity_df = pd.read_csv("d1.TOT.ELCT.ACCS.csv", skiprows=4)
gdp_df = pd.read_csv("d2.GDP.DATA.csv", skiprows=4)
education_df = pd.read_csv("d3.EDU.EXP.csv", skiprows=4)
urban_df = pd.read_csv("d4.URBAN.POP.csv", skiprows=4)
nasa_df = pd.read_excel("d5.NASA.UYDU.xlsx", sheet_name="Data")

print("Datasets loaded successfully.")


In [ ]:
# World Bank Data Preparation
def format_world_bank_data(df, value_name):
    years = [str(year) for year in range(1992, 2014)]
    df_long = df.melt(
        id_vars=["Country Code"],
        value_vars=years,
        var_name="Year",
        value_name=value_name
    )
    df_long["Year"] = df_long["Year"].astype(int)
    df_long = df_long.rename(columns={"Country Code": "ISO"})
    return df_long

national_electricity_long = format_world_bank_data(national_electricity_df, "National_Electricity_Access")
gdp_long = format_world_bank_data(gdp_df, "GDP_per_Capita")
education_long = format_world_bank_data(education_df, "Education_Expenditure")
urban_long = format_world_bank_data(urban_df, "Urban_Population")

print("World Bank datasets converted successfully.")


In [ ]:
# NASA and School Data Preparation
school_df = school_df.rename(columns={"ISO_Code": "ISO"})

nasa_selected = nasa_df[["ISO", "Year", "stbSoL", "Lpc", "Pdens", "wGini_L050"]]
school_selected = school_df[["ISO", "Year", "Electricity_Access_Percent"]]


In [ ]:
# Merge Datasets
merged_df = school_selected.copy()
for df in [nasa_selected, national_electricity_long, gdp_long, education_long, urban_long]:
    merged_df = pd.merge(merged_df, df, on=["ISO", "Year"], how="inner")

merged_df = merged_df.dropna()

print("Final dataset shape:", merged_df.shape)
merged_df.head()


In [ ]:
# Define Features and Target
X = merged_df.drop(columns=["ISO", "Year", "Electricity_Access_Percent"])
y = merged_df["Electricity_Access_Percent"]
groups = merged_df["ISO"]

print("Features:", X.columns.tolist())
print("Target:", y.name)
print("Number of countries:", groups.nunique())


In [ ]:
# Train-Test Split (Group-based to prevent data leakage)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
print(f"Train countries: {groups_train.nunique()} | Test countries: {groups.iloc[test_idx].nunique()}")


## **2. Model Evaluation**

### Initial XGBoost Model

In [ ]:
# Initial XGBoost Model
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42
)

xgb_model.fit(X_train, y_train)
xgb_predictions = xgb_model.predict(X_test)

xgb_r2   = r2_score(y_test, xgb_predictions)
xgb_mae  = mean_absolute_error(y_test, xgb_predictions)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_predictions))

print("Initial XGBoost Results")
print(f"R2 Score : {xgb_r2:.4f}")
print(f"MAE      : {xgb_mae:.4f}")
print(f"RMSE     : {xgb_rmse:.4f}")


In [ ]:
# Cross-Validation for Initial XGBoost (GroupKFold)
gkfold = GroupKFold(n_splits=3)

xgb_cv_scores = cross_val_score(
    xgb_model, X_train, y_train,
    cv=gkfold, groups=groups_train, scoring="r2"
)

print("Initial XGBoost Cross-Validation Scores:", xgb_cv_scores)
print(f"Mean CV R2 : {xgb_cv_scores.mean():.4f}")
print(f"Std Dev    : {xgb_cv_scores.std():.4f}")


## **3. Refinement Techniques**

Three refinement techniques were applied: hyperparameter tuning with GridSearchCV, cross-validation with GroupKFold to prevent data leakage, and feature importance-based feature selection.

## **4. Hyperparameter Tuning**

In [ ]:
# Hyperparameter Tuning with GridSearchCV
param_grid = {
    "n_estimators"     : [100, 200, 300],
    "learning_rate"    : [0.01, 0.05, 0.1],
    "max_depth"        : [3, 4, 5],
    "subsample"        : [0.8, 1.0],
    "colsample_bytree" : [0.8, 1.0]
}

xgb_base = XGBRegressor(random_state=42, objective="reg:squarederror")

grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=gkfold,
    scoring="r2",
    n_jobs=-1
)

grid_search.fit(X_train, y_train, groups=groups_train)

print("Best Parameters:")
print(grid_search.best_params_)
print(f"Best CV R2: {grid_search.best_score_:.4f}")


In [ ]:
# Tuned XGBoost Evaluation
tuned_xgb_model = grid_search.best_estimator_
tuned_predictions = tuned_xgb_model.predict(X_test)

tuned_r2   = r2_score(y_test, tuned_predictions)
tuned_mae  = mean_absolute_error(y_test, tuned_predictions)
tuned_rmse = np.sqrt(mean_squared_error(y_test, tuned_predictions))

# Cross-Validation for Tuned Model
tuned_cv_scores = cross_val_score(
    tuned_xgb_model, X_train, y_train,
    cv=gkfold, groups=groups_train, scoring="r2"
)

print("Tuned XGBoost Results")
print(f"R2 Score   : {tuned_r2:.4f}")
print(f"MAE        : {tuned_mae:.4f}")
print(f"RMSE       : {tuned_rmse:.4f}")
print(f"Mean CV R2 : {tuned_cv_scores.mean():.4f}")
print(f"Std Dev    : {tuned_cv_scores.std():.4f}")


In [ ]:
# Compare Initial vs Tuned XGBoost
xgb_refinement_results = pd.DataFrame({
    "Model"      : ["Initial XGBoost", "Tuned XGBoost"],
    "R2 Score"   : [xgb_r2,   tuned_r2],
    "MAE"        : [xgb_mae,  tuned_mae],
    "RMSE"       : [xgb_rmse, tuned_rmse],
    "CV Mean R2" : [xgb_cv_scores.mean(), tuned_cv_scores.mean()],
    "CV Std"     : [xgb_cv_scores.std(),  tuned_cv_scores.std()]
})

xgb_refinement_results


In [ ]:
# Comparison Graphs
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ["R2 Score", "MAE", "RMSE"]

for ax, metric in zip(axes, metrics):
    ax.bar(xgb_refinement_results["Model"], xgb_refinement_results[metric])
    ax.set_title(f"Initial vs Tuned XGBoost - {metric}")
    ax.set_ylabel(metric)
    ax.set_xlabel("Model")

plt.tight_layout()
plt.show()


## **5. Cross-Validation**

In [ ]:
# Select best model based on CV Mean R2 (more reliable than test R2 alone)
best_refinement_idx = xgb_refinement_results["CV Mean R2"].idxmax()
best_refinement_model_name = xgb_refinement_results.loc[best_refinement_idx, "Model"]

print("Best model after refinement (by CV Mean R2):", best_refinement_model_name)

if best_refinement_model_name == "Initial XGBoost":
    final_model      = xgb_model
    final_predictions = xgb_predictions
    final_cv_scores  = xgb_cv_scores
else:
    final_model      = tuned_xgb_model
    final_predictions = tuned_predictions
    final_cv_scores  = tuned_cv_scores


## **6. Feature Selection**

In [ ]:
# Feature Importance of Final Model
feature_importance = pd.DataFrame({
    "Feature"   : X.columns,
    "Importance": final_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(feature_importance.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.barh(feature_importance["Feature"], feature_importance["Importance"])
plt.gca().invert_yaxis()
plt.title(f"Feature Importance - Final XGBoost Model")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


In [ ]:
# Feature Selection (above-mean importance)
importance_mean = feature_importance["Importance"].mean()
selected_features = feature_importance[
    feature_importance["Importance"] > importance_mean
]["Feature"].tolist()

print("Selected Features:", selected_features)


In [ ]:
# Selected Feature XGBoost Model
X_train_selected = X_train[selected_features]
X_test_selected  = X_test[selected_features]

selected_xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42
)

selected_xgb_model.fit(X_train_selected, y_train)
selected_predictions = selected_xgb_model.predict(X_test_selected)

selected_r2   = r2_score(y_test, selected_predictions)
selected_mae  = mean_absolute_error(y_test, selected_predictions)
selected_rmse = np.sqrt(mean_squared_error(y_test, selected_predictions))

# Cross-Validation for Selected Feature Model
selected_cv_scores = cross_val_score(
    selected_xgb_model, X_train_selected, y_train,
    cv=gkfold, groups=groups_train, scoring="r2"
)

print("Selected Feature XGBoost Results")
print(f"R2 Score   : {selected_r2:.4f}")
print(f"MAE        : {selected_mae:.4f}")
print(f"RMSE       : {selected_rmse:.4f}")
print(f"Mean CV R2 : {selected_cv_scores.mean():.4f}")
print(f"Std Dev    : {selected_cv_scores.std():.4f}")


In [ ]:
# Final Part A Results Table (all 3 models + CV scores)
part_a_final_results = pd.DataFrame({
    "Model"      : ["Initial XGBoost", "Tuned XGBoost", "Selected Feature XGBoost"],
    "R2 Score"   : [xgb_r2,   tuned_r2,   selected_r2],
    "MAE"        : [xgb_mae,  tuned_mae,  selected_mae],
    "RMSE"       : [xgb_rmse, tuned_rmse, selected_rmse],
    "CV Mean R2" : [xgb_cv_scores.mean(), tuned_cv_scores.mean(), selected_cv_scores.mean()],
    "CV Std"     : [xgb_cv_scores.std(),  tuned_cv_scores.std(),  selected_cv_scores.std()]
})

part_a_final_results


In [ ]:
# Final Model Selection (by CV Mean R2)
best_final_idx   = part_a_final_results["CV Mean R2"].idxmax()
best_final_model_name = part_a_final_results.loc[best_final_idx, "Model"]

print("Best final model:", best_final_model_name)

if best_final_model_name == "Initial XGBoost":
    final_model       = xgb_model
    final_predictions = xgb_predictions
    final_cv_scores   = xgb_cv_scores

elif best_final_model_name == "Tuned XGBoost":
    final_model       = tuned_xgb_model
    final_predictions = tuned_predictions
    final_cv_scores   = tuned_cv_scores

else:
    final_model       = selected_xgb_model
    final_predictions = selected_predictions
    final_cv_scores   = selected_cv_scores
    X_test            = X_test_selected


In [ ]:
# Actual vs Predicted Plot
plt.figure(figsize=(7, 6))
plt.scatter(y_test, final_predictions, alpha=0.7)
min_val = min(y_test.min(), final_predictions.min())
max_val = max(y_test.max(), final_predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], color="red", linestyle="--", label="Perfect Fit")
plt.xlabel("Actual Electricity Access in Schools (%)")
plt.ylabel("Predicted Electricity Access in Schools (%)")
plt.title("Actual vs Predicted - Final XGBoost Model")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Residual Plot
residuals = y_test - final_predictions

plt.figure(figsize=(8, 5))
plt.scatter(final_predictions, residuals, alpha=0.7)
plt.axhline(y=0, color="red", linestyle="--")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residual Plot - Final XGBoost Model")
plt.tight_layout()
plt.show()


In [ ]:
# Save Part A Results
xgb_refinement_results.to_csv("xgboost_refinement_results.csv", index=False)
feature_importance.to_csv("xgboost_feature_importance.csv", index=False)
part_a_final_results.to_csv("part_a_final_results.csv", index=False)

print("Part A results saved successfully.")
print("\nFinal Model Summary:")
final_row = part_a_final_results.loc[best_final_idx]
for col in part_a_final_results.columns:
    print(f"  {col}: {final_row[col]}")


## Part A Summary



## Part A Summary

In the model comparison stage (Assignment 2), three models were evaluated — Random Forest, XGBoost, and Gradient Boosting. XGBoost achieved the best overall performance and was selected for refinement in this phase.
The first key methodological improvement was the use of GroupShuffleSplit for train-test splitting and GroupKFold for cross-validation. Since the dataset consists of multiple years of observations per country, a standard random split would place the same country in both training and test sets, causing data leakage. By splitting on country groups, each country appears exclusively in either training or test, ensuring a fair and unbiased evaluation.
Three model variants were trained and compared. The Initial XGBoost used default parameters and achieved a test R² of 0.921 and a CV Mean R² of 0.701. Hyperparameter tuning was then applied using GridSearchCV, which tested 108 parameter combinations across n_estimators, learning_rate, max_depth, subsample, and colsample_bytree. The Tuned XGBoost (n_estimators=300, learning_rate=0.01, max_depth=3, subsample=0.8, colsample_bytree=0.8) improved the test R² to 0.925 and the CV Mean R² to 0.809, with a lower standard deviation of 0.111, indicating more stable generalization. Finally, feature selection was applied based on above-mean importance scores, yielding a three-feature model (GDP per capita, National Electricity Access, Lpc) with a test R² of 0.873 and CV Mean R² of 0.765.
**The Tuned XGBoost was selected as the final model for Part B and deployment**, as it achieved the highest CV Mean R² and the most consistent performance across folds.

# **PART B: Test Submission**

## **1. Overview**

In Part A, the XGBoost model was refined through hyperparameter tuning, cross-validation, and feature selection. The Tuned XGBoost model was selected as the final model based on its superior CV Mean R² of 0.809 and test R² of 0.925.

Part B focuses on formally evaluating this final model on the held-out test set, documenting the results, and preparing the model for deployment. This phase consists of the following steps:

- **Data Preparation**: Verifying that the test data is in the correct format for the model.
- **Model Application**: Applying the final model to the test set to generate predictions.
- **Test Metrics**: Evaluating prediction quality using R², MAE, and RMSE, and comparing with Part A results.
- **Model Deployment**: Serializing the trained model using joblib so it can be reused without retraining.
- **Code Implementation**: Building a simple Flask API that accepts feature inputs and returns a prediction.

## **2. Data Preparation**

The test set was already created in Part A using GroupShuffleSplit, which ensures that countries in the test set were not seen during training. Here we verify the test data format and confirm it is ready for model application.

In [ ]:
# Verify test data
print("Test set shape:", X_test.shape)
print("Test set features:", X_test.columns.tolist())
print("Number of test countries:", groups.iloc[test_idx].nunique())
print("\nSample test data:")
X_test.head()


In [ ]:
# Check for missing values in test set
missing = X_test.isnull().sum()
print("Missing values in test set:")
print(missing)
print("\nTest target distribution:")
print(y_test.describe())


## **3. Model Application**

The Tuned XGBoost model selected in Part A is applied to the test set to generate predictions. These predictions represent the model's estimate of school electricity access percentage for unseen countries.

In [ ]:
# Apply final model to test set
test_predictions = final_model.predict(X_test)

# Create results dataframe
test_results = pd.DataFrame({
    "Country"   : groups.iloc[test_idx].values,
    "Actual"    : y_test.values,
    "Predicted" : test_predictions.round(2),
    "Error"     : (y_test.values - test_predictions).round(2)
})

print("Predictions on test set:")
test_results


## **4. Test Metrics**

The model's performance is evaluated using three metrics: R² (explained variance), MAE (mean absolute error), and RMSE (root mean squared error). Results are compared against Part A cross-validation scores to confirm consistency.

In [ ]:
# Calculate test metrics
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

test_r2   = r2_score(y_test, test_predictions)
test_mae  = mean_absolute_error(y_test, test_predictions)
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))

print("=" * 40)
print("Final Model Test Results")
print("=" * 40)
print(f"R2 Score : {test_r2:.4f}")
print(f"MAE      : {test_mae:.4f}")
print(f"RMSE     : {test_rmse:.4f}")
print("=" * 40)
print(f"CV Mean R2 (from Part A): {final_cv_scores.mean():.4f}")
print(f"CV Std     (from Part A): {final_cv_scores.std():.4f}")


In [ ]:
# Comparison table: Part A CV vs Part B Test
comparison = pd.DataFrame({
    "Metric"        : ["R2 Score", "MAE", "RMSE"],
    "Part A CV Mean": [
        round(final_cv_scores.mean(), 4),
        "-",
        "-"
    ],
    "Part B Test"   : [
        round(test_r2, 4),
        round(test_mae, 4),
        round(test_rmse, 4)
    ]
})

print(comparison.to_string(index=False))


In [ ]:
# Actual vs Predicted Plot
plt.figure(figsize=(7, 6))
plt.scatter(y_test, test_predictions, alpha=0.7, color="steelblue")
min_val = min(y_test.min(), test_predictions.min())
max_val = max(y_test.max(), test_predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], color="red", linestyle="--", label="Perfect Fit")
plt.xlabel("Actual Electricity Access (%)")
plt.ylabel("Predicted Electricity Access (%)")
plt.title("Actual vs Predicted - Test Set")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Residual Plot
residuals = y_test.values - test_predictions

plt.figure(figsize=(8, 5))
plt.scatter(test_predictions, residuals, alpha=0.7, color="steelblue")
plt.axhline(y=0, color="red", linestyle="--")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residual Plot - Test Set")
plt.tight_layout()
plt.show()


In [ ]:
# Error distribution
plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=15, color="steelblue", edgecolor="white")
plt.axvline(x=0, color="red", linestyle="--")
plt.xlabel("Prediction Error")
plt.ylabel("Frequency")
plt.title("Distribution of Prediction Errors - Test Set")
plt.tight_layout()
plt.show()

print(f"\nMean Error  : {residuals.mean():.4f}")
print(f"Std of Error: {residuals.std():.4f}")


## **5. Model Deployment**

The trained model is serialized using `joblib` and saved as a `.pkl` file. This allows the model to be loaded and used in other environments (e.g., APIs, other notebooks, production systems) without retraining.

In [ ]:
import joblib

# Save the final model
joblib.dump(final_model, "tuned_xgb_model.pkl")

# Save feature list for API use
feature_list = X_test.columns.tolist()
joblib.dump(feature_list, "model_features.pkl")

print("Model saved as: tuned_xgb_model.pkl")
print("Features saved as: model_features.pkl")
print("Features:", feature_list)


In [ ]:
# Verify the saved model loads and predicts correctly
loaded_model    = joblib.load("tuned_xgb_model.pkl")
loaded_features = joblib.load("model_features.pkl")

sample_input    = X_test.iloc[[0]]
original_pred   = final_model.predict(sample_input)[0]
loaded_pred     = loaded_model.predict(sample_input)[0]

print("Verification: saved model produces identical predictions")
print(f"Original model prediction : {original_pred:.4f}")
print(f"Loaded model prediction   : {loaded_pred:.4f}")
print(f"Match: {abs(original_pred - loaded_pred) < 1e-6}")


## **6. Code Implementation**

A Flask API is implemented to serve the model. The API accepts a JSON object containing the 8 feature values and returns the predicted school electricity access percentage. The API runs in a background thread so it can be tested directly within the notebook.

In [ ]:
from flask import Flask, request, jsonify
import joblib
import threading

app = Flask(__name__)

_model    = joblib.load("tuned_xgb_model.pkl")
_features = joblib.load("model_features.pkl")

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()

    missing = [f for f in _features if f not in data]
    if missing:
        return jsonify({"error": f"Missing features: {missing}"}), 400

    try:
        values     = [[data[f] for f in _features]]
        prediction = _model.predict(values)[0]
        return jsonify({
            "prediction" : round(float(prediction), 2),
            "unit"       : "% electricity access in schools",
            "model"      : "Tuned XGBoost",
            "features_used": _features
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": "Tuned XGBoost"})

def run_api():
    app.run(port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_api)
thread.daemon = True
thread.start()

print("API is running at http://localhost:5000")
print("Required features:", _features)


In [ ]:
import requests

# Test with first sample from test set
sample = X_test.iloc[0].to_dict()

print("Input features:")
for k, v in sample.items():
    print(f"  {k}: {v:.4f}")

response = requests.post("http://localhost:5000/predict", json=sample)
result   = response.json()

print("\nAPI Response:")
print(f"  Predicted : {result['prediction']}%")
print(f"  Actual    : {y_test.iloc[0]:.2f}%")
print(f"  Error     : {abs(result['prediction'] - y_test.iloc[0]):.2f}%")


In [ ]:
# Save test results to CSV
test_results.to_csv("part_b_test_results.csv", index=False)

print("Part B results saved to: part_b_test_results.csv")
print("\nPart B Summary:")
print(f"  Final Model  : Tuned XGBoost")
print(f"  Test R2      : {test_r2:.4f}")
print(f"  Test MAE     : {test_mae:.4f}")
print(f"  Test RMSE    : {test_rmse:.4f}")
print(f"  CV Mean R2   : {final_cv_scores.mean():.4f}")


## **Part B Summary**



In Part B, the Tuned XGBoost model selected in Part A was formally evaluated on the held-out test set. The test set consisted of 50 observations from 11 countries that were not seen during training, ensuring an unbiased evaluation.

The model achieved a test R² of 0.925, MAE of 2.071, and RMSE of 5.692. These results are consistent with the CV Mean R² of 0.809 obtained in Part A, confirming that the model generalizes well to unseen countries and is not overfitting.

The model was serialized using joblib and saved as a .pkl file, making it portable and reusable without retraining. A Flask API was implemented to serve real-time predictions, accepting the 8 input features and returning the predicted school electricity access percentage.





# **PART C: Deployment**

## **1. Overview**

Part C extends the basic API from Part B into a production-ready deployment. The model artifact is documented with full metadata, the API is rebuilt with input validation and structured error handling, security measures are added, and every request is logged for monitoring purposes.

## **2. Model Serialization**

The model saved in Part B is loaded and documented with metadata including version, hyperparameters, performance metrics, and training information.

In [ ]:
import joblib
import json
import os
from datetime import datetime

# Load saved model and features
model    = joblib.load("tuned_xgb_model.pkl")
features = joblib.load("model_features.pkl")

# Create metadata
model_metadata = {
    "model_name"      : "Tuned XGBoost",
    "version"         : "1.0.0",
    "saved_at"        : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "target_variable" : "Electricity_Access_Percent",
    "features"        : features,
    "n_features"      : len(features),
    "hyperparameters" : {
        "n_estimators"     : 300,
        "learning_rate"    : 0.01,
        "max_depth"        : 3,
        "subsample"        : 0.8,
        "colsample_bytree" : 0.8
    },
    "performance": {
        "test_r2"    : 0.9245,
        "test_mae"   : 2.0712,
        "test_rmse"  : 5.6924,
        "cv_mean_r2" : 0.8089,
        "cv_std"     : 0.1114
    },
    "training_info": {
        "train_size"   : 173,
        "test_size"    : 50,
        "n_countries"  : 53,
        "split_method" : "GroupShuffleSplit",
        "cv_method"    : "GroupKFold (n_splits=3)"
    }
}

with open("model_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=2)

print("Model Metadata:")
print(json.dumps(model_metadata, indent=2))


## **3. Model Serving**

The Flask API is built with three endpoints: `/predict` for predictions, `/health` for status checks, and `/metadata` for model information. All prediction requests are validated before being passed to the model. The API runs on port 5001 to avoid conflicts with Part B.

In [ ]:
from flask import Flask, request, jsonify
import threading
import time

# In-memory log storage (avoids file path issues in Colab)
request_logs = []

app2 = Flask("deployment_api")

_model    = joblib.load("tuned_xgb_model.pkl")
_features = joblib.load("model_features.pkl")
_metadata = json.load(open("model_metadata.json"))

# Valid input ranges based on training data
FEATURE_RANGES = {
    "stbSoL"                     : (0, 1e9),
    "Lpc"                        : (0, 10),
    "Pdens"                      : (0, 5000),
    "wGini_L050"                 : (0, 1),
    "National_Electricity_Access": (0, 100),
    "GDP_per_Capita"             : (0, 200000),
    "Education_Expenditure"      : (0, 100),
    "Urban_Population"           : (0, 100)
}

VALID_API_KEYS = {"assignment3-key-2024"}

def validate_api_key(req):
    return req.headers.get("X-API-Key") in VALID_API_KEYS

def validate_inputs(data):
    errors = []
    for feature in _features:
        if feature not in data:
            errors.append(f"Missing feature: {feature}")
            continue
        val = data[feature]
        if not isinstance(val, (int, float)):
            errors.append(f"Invalid type for {feature}: expected number")
            continue
        low, high = FEATURE_RANGES[feature]
        if not (low <= val <= high):
            errors.append(f"{feature}={val} is out of range [{low}, {high}]")
    return errors

@app2.route("/predict", methods=["POST"])
def predict():
    start_time = time.time()

    if not validate_api_key(request):
        return jsonify({"error": "Unauthorized. Provide a valid X-API-Key header."}), 401

    data = request.get_json()
    if not data:
        return jsonify({"error": "No JSON body provided"}), 400

    errors = validate_inputs(data)
    if errors:
        return jsonify({"error": "Input validation failed", "details": errors}), 400

    values     = [[data[f] for f in _features]]
    prediction = _model.predict(values)[0]
    duration   = round(time.time() - start_time, 4)

    # Store log in memory
    request_logs.append({
        "timestamp" : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "inputs"    : data,
        "prediction": round(float(prediction), 2),
        "duration_s": duration,
        "status"    : 200
    })

    return jsonify({
        "prediction"   : round(float(prediction), 2),
        "unit"         : "% electricity access in schools",
        "model"        : _metadata["model_name"],
        "version"      : _metadata["version"],
        "response_time": f"{duration}s"
    })

@app2.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status"  : "ok",
        "model"   : _metadata["model_name"],
        "version" : _metadata["version"],
        "features": _features
    })

@app2.route("/metadata", methods=["GET"])
def metadata():
    return jsonify(_metadata)

def run_api2():
    app2.run(port=5001, debug=False, use_reloader=False)

thread2 = threading.Thread(target=run_api2)
thread2.daemon = True
thread2.start()

print("Deployment API running at http://localhost:5001")
print("Endpoints:")
print("  POST /predict  — Make a prediction (requires X-API-Key header)")
print("  GET  /health   — Check API status")
print("  GET  /metadata — View model metadata")


## **4. API Integration**

Five test scenarios verify that the API handles all cases correctly: valid input, missing API key, missing feature, out-of-range value, and health check.

In [ ]:
import requests

API_URL = "http://localhost:5001"
HEADERS = {"X-API-Key": "assignment3-key-2024"}
sample  = X_test.iloc[0].to_dict()

print("=" * 50)
print("Scenario 1: Valid request")
print("=" * 50)
r = requests.post(f"{API_URL}/predict", json=sample, headers=HEADERS)
print(f"Status  : {r.status_code}")
print(f"Response: {r.json()}")


In [ ]:
print("=" * 50)
print("Scenario 2: Missing API key")
print("=" * 50)
r = requests.post(f"{API_URL}/predict", json=sample)
print(f"Status  : {r.status_code}")
print(f"Response: {r.json()}")


In [ ]:
print("=" * 50)
print("Scenario 3: Missing feature")
print("=" * 50)
incomplete = {k: v for k, v in sample.items() if k != "GDP_per_Capita"}
r = requests.post(f"{API_URL}/predict", json=incomplete, headers=HEADERS)
print(f"Status  : {r.status_code}")
print(f"Response: {r.json()}")


In [ ]:
print("=" * 50)
print("Scenario 4: Out-of-range value")
print("=" * 50)
bad_input = sample.copy()
bad_input["GDP_per_Capita"] = -500
r = requests.post(f"{API_URL}/predict", json=bad_input, headers=HEADERS)
print(f"Status  : {r.status_code}")
print(f"Response: {r.json()}")


In [ ]:
print("=" * 50)
print("Scenario 5: Health check")
print("=" * 50)
r = requests.get(f"{API_URL}/health")
print(f"Status  : {r.status_code}")
print(f"Response: {r.json()}")


## **5. Security Considerations**

Three security layers are implemented to protect the API from unauthorized access and invalid inputs.

In [ ]:
security_summary = pd.DataFrame({
    "Security Measure"       : [
        "API Key Authentication",
        "Input Type Validation",
        "Input Range Validation"
    ],
    "Purpose"                : [
        "Restricts access to authorized users only",
        "Ensures all feature values are numeric",
        "Prevents unrealistic or malicious values"
    ],
    "Implementation"         : [
        "X-API-Key header checked on every request",
        "isinstance() check for int/float on each feature",
        "Min/max bounds defined per feature from training data"
    ],
    "HTTP Response if Failed": [
        "401 Unauthorized",
        "400 Bad Request",
        "400 Bad Request"
    ]
})

print(security_summary.to_string(index=False))


## **6. Monitoring & Logging**

Every prediction request is stored in memory with timestamp, inputs, prediction, and response time. This allows tracking of model usage and detection of unusual inputs over time.

In [ ]:
# Generate a few more requests to populate logs
for i in range(3):
    requests.post(f"{API_URL}/predict", json=X_test.iloc[i].to_dict(), headers=HEADERS)

# Display logs
print(f"Total requests logged: {len(request_logs)}")
print("-" * 60)
for entry in request_logs:
    print(f"[{entry['timestamp']}] prediction={entry['prediction']}% | duration={entry['duration_s']}s")


In [ ]:
# Log summary statistics
if request_logs:
    log_df = pd.DataFrame([{
        "prediction": e["prediction"],
        "duration_s": e["duration_s"]
    } for e in request_logs])

    print("Log Summary:")
    print(f"  Total requests    : {len(log_df)}")
    print(f"  Avg prediction    : {log_df['prediction'].mean():.2f}%")
    print(f"  Min prediction    : {log_df['prediction'].min():.2f}%")
    print(f"  Max prediction    : {log_df['prediction'].max():.2f}%")
    print(f"  Avg response time : {log_df['duration_s'].mean():.4f}s")

    plt.figure(figsize=(8, 4))
    plt.hist(log_df["prediction"], bins=10, color="steelblue", edgecolor="white")
    plt.xlabel("Predicted Electricity Access (%)")
    plt.ylabel("Frequency")
    plt.title("Distribution of Predictions — Logged Requests")
    plt.tight_layout()
    plt.show()


## **Part C Summary**

**English:**

Part C extended the basic API from Part B into a production-ready deployment. The model was documented with full metadata including version, hyperparameters, and performance metrics. The Flask API was rebuilt with three endpoints (`/predict`, `/health`, `/metadata`) running on port 5001.

Security was implemented through API key authentication (returns 401 on failure), input type validation, and feature range validation (returns 400 on failure). Monitoring was handled through in-memory logging, recording each request's timestamp, inputs, prediction, and response time. Five integration test scenarios confirmed correct API behavior under valid and invalid conditions.

---

**Türkçe:**

Part C'de, Part B'deki temel API production ortamına hazır hale getirildi. Model; versiyon, hiperparametreler ve performans metrikleri içeren tam metadata ile belgelendi. Flask API, `/predict`, `/health` ve `/metadata` olmak üzere üç endpoint ile 5001 portunda yeniden yapılandırıldı.

Güvenlik katmanı olarak API key doğrulaması (401), girdi tipi kontrolü ve özellik aralığı doğrulaması (400) eklendi. İzleme ise her isteğin zaman damgası, girdiler, tahmin ve yanıt süresiyle birlikte bellekte loglanması yoluyla sağlandı. Beş test senaryosu ile API'nin geçerli ve geçersiz durumlarda doğru davrandığı doğrulandı.